# 01 · Dataset preparation & text → embedding pipeline

**Milestone 1, From Words to Worlds**

1. Download COCO Captions (val2017: 5,000 images × ~5 captions)
2. Clean captions, take a subset, resize/center-crop images to 512×512, normalize to [-1, 1]
3. Encode captions with a Transformer text encoder (CLIP ViT-L/14, the same encoder Stable Diffusion v1.5 conditions on)
4. Sanity-check the embeddings: caption–caption similarity and CLIP text→image retrieval

> Runtime: GPU (Colab → Runtime → Change runtime type → T4 GPU).

In [ ]:
# --- Environment setup (Colab or university GPU) ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/words-to-worlds.git"  # TODO: set after pushing
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True  # Colab only: keep data/ and outputs/ on Drive so they survive restarts

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        workdir = Path("/content/drive/MyDrive/words-to-worlds")
    else:
        workdir = Path("/content/words-to-worlds")
    if not workdir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(workdir)], check=True)
    os.chdir(workdir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # On the cluster, launch Jupyter from the repo root (or cd there)
    while not Path("pyproject.toml").exists() and Path.cwd() != Path("/"):
        os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(Path.cwd() / "src"))

import torch
print("cwd:", Path.cwd())
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Download, clean, subset, preprocess
Downloads ~1.2 GB (val2017 images + annotations) once. Set `NUM_IMAGES` between 1000 and 5000.

In [ ]:
NUM_IMAGES = 5000
!python scripts/prepare_data.py --num-images {NUM_IMAGES}

In [ ]:
import json
from t2i.config import load_config, resolve
from t2i.data import CocoCaptionDataset, read_jsonl, to_pil

cfg = load_config()
PROC = resolve(cfg["data"]["processed"])
print(json.dumps(json.loads((PROC / "stats.json").read_text()), indent=2))
records = read_jsonl(PROC / "captions.jsonl")
ds = CocoCaptionDataset(PROC)
sample = ds[0]
print("pixel tensor:", tuple(sample["pixel_values"].shape), "range",
      round(sample["pixel_values"].min().item(), 2), round(sample["pixel_values"].max().item(), 2))

### Look at the data

In [ ]:
import matplotlib.pyplot as plt, random, textwrap
random.seed(0)
idx = random.sample(range(len(ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, i in zip(axes.flat, idx):
    item = ds[i]
    ax.imshow(to_pil(item["pixel_values"])); ax.axis("off")
    caps = records[i]["captions"][:3]
    ax.set_title("\n".join(textwrap.shorten(c, 45) for c in caps), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
lengths = [len(c.split()) for r in records for c in r["captions"]]
plt.figure(figsize=(6, 3))
plt.hist(lengths, bins=range(0, 30)); plt.xlabel("words per caption"); plt.ylabel("count")
plt.title("COCO caption lengths (all fit in CLIP's 77-token window)"); plt.show()

## 2. Text → embedding pipeline

CLIP's text transformer maps a caption to:
- **per-token hidden states** `(B, 77, 768)`: what Stable Diffusion's U-Net cross-attends to (`prompt_embeds`)
- **pooled embedding** `(B, 768)`: the EOS token, used for similarity / retrieval

In [ ]:
from t2i.embeddings import TextEncoder
te_cfg = cfg["text_encoder"]
enc = TextEncoder(te_cfg["name"], te_cfg["type"], te_cfg["max_length"])

caption = records[0]["captions"][0]
tok = enc.tokenize([caption])
n_real = int(tok["attention_mask"].sum())
print("caption:", caption)
print("tokens :", enc.tokenizer.convert_ids_to_tokens(tok["input_ids"][0][:n_real]))
out = enc.encode([caption])
print("hidden :", tuple(out["hidden"].shape), "| pooled:", tuple(out["pooled"].shape))

### Do captions of the same image land close together?
Cosine similarity between all captions of 6 images: expect bright blocks on the diagonal.

In [ ]:
import torch, torch.nn.functional as F
sub = records[:6]
texts = [c for r in sub for c in r["captions"]]
owner = [k for k, r in enumerate(sub) for _ in r["captions"]]
z = F.normalize(enc.encode(texts)["pooled"].float(), dim=-1).cpu()
sim = z @ z.T
plt.figure(figsize=(6, 5)); plt.imshow(sim, cmap="viridis"); plt.colorbar()
plt.title("CLIP caption–caption cosine similarity"); plt.xlabel("caption"); plt.ylabel("caption"); plt.show()

same = torch.tensor([[a == b for b in owner] for a in owner])
off = ~torch.eye(len(texts), dtype=bool)
print(f"mean sim, same image: {sim[same & off].mean():.3f} | different image: {sim[~same].mean():.3f}")

### Cross-modal check: CLIP text → image retrieval
Embed 500 images with CLIP's image tower and retrieve each image from its first caption. Good Recall@K shows text and image share one embedding space, which is what makes text conditioning work.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
dev = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained(te_cfg["name"], torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
proc = CLIPProcessor.from_pretrained(te_cfg["name"])

N = 500
recs = records[:N]
img_emb, txt_emb = [], []
with torch.no_grad():
    for i in range(0, N, 50):
        batch = recs[i:i+50]
        imgs = [Image.open(PROC / "images" / r["file_name"]).convert("RGB") for r in batch]
        px = proc(images=imgs, return_tensors="pt")["pixel_values"].to(dev, clip.dtype)
        img_emb.append(F.normalize(clip.get_image_features(pixel_values=px).float(), dim=-1))
        t = proc(text=[r["captions"][0] for r in batch], return_tensors="pt", padding=True, truncation=True).to(dev)
        txt_emb.append(F.normalize(clip.get_text_features(**t).float(), dim=-1))
img_emb, txt_emb = torch.cat(img_emb), torch.cat(txt_emb)
ranks = (txt_emb @ img_emb.T).argsort(dim=1, descending=True)
gt = torch.arange(N, device=ranks.device).unsqueeze(1)
for k in (1, 5, 10):
    print(f"text→image Recall@{k}: {(ranks[:, :k] == gt).any(1).float().mean():.3f}  (chance {k/N:.3f})")
del clip; torch.cuda.empty_cache() if dev == "cuda" else None

## 3. Cache embeddings for all captions
Saves pooled CLIP embeddings for every caption to `data/processed/embeddings_clip-vit-large-patch14.pt`; later milestones reuse them.

In [ ]:
!python scripts/encode_captions.py

In [ ]:
emb = torch.load(PROC / "embeddings_clip-vit-large-patch14.pt")
print(emb["pooled"].shape, emb["pooled"].dtype, "| first:", emb["texts"][0])